# Lab Exercise: The End-to-End Machine Learning Pipeline
This notebook implements the seven-step machine learning pipeline sequentially on the Iris flower dataset.
You will see how a raw dataset is loaded, partitioned, scaled, fitted, predicted, evaluated, and exported as a serialized asset.

We will then condense the entire pipeline into a scikit-learn `Pipeline` object to remove data leakage risks and swap out multiple classification algorithms in a single line.

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ---------- STEP 1: Preprocess ----------
iris = load_iris(as_frame=True)
df = iris.frame

print("STEP 1: PREPROCESS")
print(f"   loaded {df.shape[0]} rows x {df.shape[1]} columns (pandas DataFrame)")
print(f"   missing values: {int(df.isna().sum().sum())}")

X = df[iris.feature_names]
y = df["target"]

print(f"   X matrix shape: {X.shape} (2-D)")
print(f"   y vector shape: {y.shape} (1-D)")
print(f"   classes: {dict(zip(map(str, iris.target_names), map(int, np.bincount(y))))}")

### Step 2: Split and Scale
We split our data into an 80% training set and a 20% test set, ensuring stratified class balances.
**Crucial rule:** The scaler MUST learn characteristics (`fit`) on the training data only, and simply apply them (`transform`) to the test set to avoid data leakage!

In [ ]:
# ---------- STEP 2: Split ----------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y)

print("STEP 2: TRAIN / TEST SPLIT")
print(f"   train: {len(X_tr)} rows, test: {len(X_te)} rows (80/20 split)")

# Fit scaler on train only
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

print(f"   scaler fitted on train only; scaled train mean: {X_tr_s.mean():.2e}, std: {X_tr_s.std():.3f}")

### Steps 3 & 4: Algorithm Setup and Model Training (Fit)

In [ ]:
# ---------- STEP 3: Set up algorithm ----------
model = LogisticRegression(max_iter=1000)
print("STEP 3: ALGORITHM SETUP")
print(f"   {model.__class__.__name__} created — currently untrained.\n")

# ---------- STEP 4: Fit (Train) ----------
model.fit(X_tr_s, y_tr)
print("STEP 4: TRAIN THE MODEL")
print(f"   fitted; learned {model.coef_.size} coefficients + {model.intercept_.size} intercepts")

### Steps 5 & 6: Prediction and Evaluation

In [ ]:
# ---------- STEP 5: Predict ----------
y_pred = model.predict(X_te_s)
print("STEP 5: PREDICT")
print(f"   first 10 predicted classes: {[int(v) for v in y_pred[:10]]}")
print(f"   first 10 actual classes:    {[int(v) for v in y_te[:10]]}\n")

# ---------- STEP 6: Evaluate ----------
print("STEP 6: EVALUATE")
print(f"   test accuracy: {accuracy_score(y_te, y_pred):.3f}")
print(classification_report(y_te, y_pred, target_names=iris.target_names))

### Step 7: Export (Save) the Model Pipeline

In [ ]:
# ---------- STEP 7: Export ----------
path = "iris_model.joblib"
joblib.dump({"scaler": scaler, "model": model}, path)
print("STEP 7: SAVE THE MODEL")
print(f"   written to {path} ({os.path.getsize(path)} bytes)")

# Reload and double check
loaded = joblib.load(path)
reloaded_pred = loaded["model"].predict(loaded["scaler"].transform(X_te))
print(f"   reloaded and re-predicted; predictions identical: {np.array_equal(y_pred, reloaded_pred)}")

# Clean up local file
if os.path.exists(path):
    os.remove(path)

### Streamlining with scikit-learn Pipeline
By tying preprocessors and estimators into a single object, we can train and predict on raw features directly, preventing any manual data leakage.

In [ ]:
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])
pipe.fit(X_tr, y_tr)
print(f"Condensated Pipeline Accuracy: {accuracy_score(y_te, pipe.predict(X_te)):.3f}")

### Easy Algorithm Swapping
Since all estimators share the same scikit-learn contract, swapping them is a one-line substitution!

In [ ]:
algorithms = [
    LogisticRegression(max_iter=1000),
    KNeighborsClassifier(n_neighbors=5),
    RandomForestClassifier(random_state=0),
    DecisionTreeClassifier(max_depth=1, random_state=0)  # Crippled tree
]

print(f"   {'Algorithm':<25} {'Accuracy':>10}")
print("-" * 40)
for algo in algorithms:
    p = Pipeline([
        ("scale", StandardScaler()),
        ("clf", algo)
    ]).fit(X_tr, y_tr)
    print(f"   {algo.__class__.__name__:<25} {accuracy_score(y_te, p.predict(X_te)):>10.3f}")

### Lab Experiments (Things to Try)

#### Experiment 1: Induce Data Leakage
1. Fit the `StandardScaler` on the entire `X` instead of `X_tr`.
2. Transform `X_tr` and `X_te` separately.
3. Fit the model and observe the accuracy shift. This represents leakage of testing bounds into training, invalidating accuracy scores on actual unseen data!

#### Experiment 2: Skip Feature Scaling
1. Remove `StandardScaler` from the pipeline, training the algorithms directly on raw features `X_tr`.
2. Look at how algorithms like KNN or Logistic Regression change performance. (Hint: KNN utilizes Euclidean distance, making it highly sensitive to unscaled feature ranges!).

#### Experiment 3: Tree Depths
1. Change `max_depth` on the `DecisionTreeClassifier` from `1` to `3`.
2. See how accuracy recovers as the tree develops more decision boundaries!